# ODI to Databricks Migration: TRG_EMP_INSERT

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook processes a full load insert into the `TRG_EMP` target table from the `EMPLOYEES` source table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Target Table: TRG_EMP

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: (No SQL provided in original, placeholder)
DROP TABLE IF EXISTS workspace.hr.trg_emp;

In [ ]:
%sql
-- SCEN_TASK_NO in {20}: (No SQL provided in original, placeholder)
CREATE TABLE workspace.hr.trg_emp (
    EMPLOYEE_ID     BIGINT,
    FIRST_NAME      STRING,
    LAST_NAME       STRING,
    EMAIL           STRING,
    PHONE_NUMBER    STRING,
    HIRE_DATE       TIMESTAMP,
    JOB_ID          STRING,
    SALARY          DECIMAL(8,2),
    COMMISSION_PCT  DECIMAL(2,2),
    MANAGER_ID      BIGINT,
    DEPARTMENT_ID   BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Insert into target table
INSERT INTO workspace.hr.trg_emp
  (
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID
  )
SELECT
  employees.employee_id,
  employees.first_name,
  employees.last_name,
  employees.email,
  employees.phone_number,
  employees.hire_date,
  employees.job_id,
  employees.salary,
  employees.commission_pct,
  employees.manager_id,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_emp;

## Conversion Notes

- **Schema Mapping**: `HR` schema was mapped to `workspace.hr`.
- **Table Naming**: `TRG_EMP` and `EMPLOYEES` were converted to lowercase `trg_emp` and `employees`.
- **Oracle Hints**: `/*+ APPEND PARALLEL */` was removed as it's not applicable to Databricks Delta Lake.
- **Data Type Mapping**: Oracle `NUMBER(p,0)` was mapped to `BIGINT`, `NUMBER(p,s)` to `DECIMAL(p,s)`, `DATE` to `TIMESTAMP`, `VARCHAR2` to `STRING`.
- **DDL Creation**: A `CREATE TABLE` statement for `TRG_EMP` was inferred and added based on the `SELECT` list from the source `EMPLOYEES` table.